# PasarPulse-MM — Main Data Ingestion Pipeline

Notebook menjalankan **public reference preparation → PIHPS/SP2KP ingestion → price-only baselines → BPS/weather/Sentinel → graph → asynchronous merge → QA → visualisasi → model-ready artifacts**.

MVP menggunakan harga PIHPS sebagai anchor dengan unit `province × commodity × market_level × day`; raw monthly responses are retained as versioned snapshots. PIHPS currently returns 34 provinces, so the mapping audit keeps the four absent Papua expansion provinces visible.

## 0. Instalasi awal

Jalankan cell instalasi hanya pada environment/VM baru. Setelah instalasi selesai, restart kernel apabila diminta.

In [1]:
# Uncomment on the first run in a new VM:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


## 1. Resolve project root dan import pipeline

Notebook diasumsikan berada di folder utama project. Cell ini juga tetap bekerja jika notebook dipindahkan satu tingkat ke folder `notebooks/`.

In [2]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config.yaml").exists() and (PROJECT_ROOT.parent / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pasarpulse import PasarPulsePipeline, credential_status, load_config

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: <PROJECT_ROOT>


## 2. Credential check

Nilai secret tidak pernah ditampilkan. `setup_credentials.py` harus berada di project root.

In [3]:
credential_checks = credential_status(PROJECT_ROOT)
display(pd.DataFrame([
    {"credential": key, "available": value}
    for key, value in credential_checks.items()
]))

,credential,available
0,CDS_API_KEY,True
1,CDSE_S3_ACCESS_KEY,True
2,CDSE_S3_SECRET_KEY,True
3,CHROME_MAJOR_VERSION,False


## 3. Load konfigurasi dan preflight

Default config memakai sumber harga publik **PIHPS** pada tiga level (retail tradisional, wholesale, producer). SP2KP tetap tersedia sebagai fallback/benchmark. Reference polygons dan satellite fallback zones dibuat otomatis dari BIG.

In [4]:
config = load_config(PROJECT_ROOT / "config.yaml", project_root=PROJECT_ROOT)
pipeline = PasarPulsePipeline(config)

display(pd.DataFrame([
    {"stage": name, "enabled": enabled}
    for name, enabled in config.section("stages").items()
]))
display(pipeline.preflight())

,stage,enabled
0,references,True
1,prices,True
2,price_baselines,True
3,bps,True
4,era5_download,True
5,era5_aggregate,True
6,sentinel_search,True
7,sentinel_extract,True
8,calendar,True
9,static_graph,True


,item,required_for,available,auto_preparable
0,setup_credentials.py,ERA5 and Sentinel-2,True,NaN
1,CDS API token,ERA5 download,True,NaN
2,CDSE S3 keys,Sentinel feature extraction,True,NaN
3,/pasar...,Weather aggregation and static graph,True,True
4,/pasar...,Sentinel search/extraction,True,True
5,/pasar...,Official calendar events,True,False
6,price source: pihps,Price anchor and full merge,True,True


## 4. Prepare public geospatial references

Mengunduh polygon 38 provinsi dari BIG dan membuat `satellite_zones.geojson` sebagai **province fallback** untuk smoke test. Untuk eksperimen final, fallback ini nanti diganti producing-district cropland zones.

In [5]:
reference_outputs = pipeline.run_references()
print(reference_outputs)
display(pipeline.preflight())

KeyboardInterrupt: 

## 5. PIHPS public price ingestion dan eligibility audit

Stage ini mengambil report resmi PIHPS per komoditas dan market level, menyimpan response JSON bulanan dengan checksum/version, mem-parsing province rows dan daily columns, lalu membentuk canonical panel. Mapping, unit, source version, raw snapshot manifest, dan coverage/eligibility tetap terpisah.

In [6]:
prices = pipeline.run_prices()
if prices is not None:
    display(prices.head())
    display(pipeline.frames["price_coverage_audit"].head(20))
    if "pihps_manifest" in pipeline.frames:
        display(pipeline.frames["pihps_manifest"][["commodity_code", "market_level", "requested_start_date", "requested_end_date", "status", "parsed_rows"]].tail(20))
        display(pipeline.frames["pihps_mapping"].query("mapping_status != 'observed'"))

KeyboardInterrupt: 

## 5b. Price-only rolling-origin baselines

Eligible series are scored with fixed last-price and same-horizon seasonal-lag baselines for 7- and 14-day forecasts. Forecasts use only observations available at the origin; this stage does not tune on future labels.

In [ ]:
baseline_metrics = pipeline.run_price_baselines()
if baseline_metrics is not None:
    display(baseline_metrics)

## 6. BPS annual production context

Stage ini memakai browser-rendered public Statistics Table dan tidak membutuhkan BPS API key. Chrome harus tersedia pada VM.

In [ ]:
production = pipeline.run_bps()
if production is not None:
    display(production.head(20))

## 7. ERA5-Land weather

`era5_download` dimatikan secara default karena ukuran data besar. Setelah diaktifkan, data diunduh per bulan sehingga dapat di-resume. Aggregation memerlukan `indonesia_provinces.geojson`.

In [ ]:
era5_files = pipeline.run_era5_download()
weather = pipeline.run_era5_aggregate()
if weather is not None:
    display(weather.head())

## 8. Sentinel-2 L2A

Search menggunakan official CDSE STAC. Feature extraction membaca B04, B08, optional B03, dan SCL melalui S3 credentials. Gunakan producing-district/cropland zones bila tersedia.

In [ ]:
satellite_manifest = pipeline.run_sentinel_search()
satellite = pipeline.run_sentinel_extract()
if satellite_manifest is not None:
    display(satellite_manifest.head())
if satellite is not None:
    display(satellite.head())

## 9. Calendar features

In [ ]:
calendar_features = pipeline.run_calendar()
if calendar_features is not None:
    display(calendar_features.head())

## 9. Static dan dynamic graph

- Static: border adjacency, centroid distance, production-capacity similarity, dan nearest-neighbour fallback untuk kepulauan.
- Dynamic: rolling price dependence, source lead-lag, shock co-occurrence, dan top-k outgoing edges.

Semua structural edges tetap candidate connectivity, bukan klaim observed commodity flow.

In [ ]:
static_graph = pipeline.run_static_graph()
dynamic_graph = pipeline.run_dynamic_graph()
if static_graph is not None:
    display(static_graph.head())
if dynamic_graph is not None:
    display(dynamic_graph.head())

## 10. Asynchronous merge dan model-ready data

Merge membuat daily grid per observed series, mempertahankan raw missingness, menerapkan bounded carry-forward, menambahkan modality masks, freshness, evidence-quality score, dan candidate labels.

In [ ]:
model_data = pipeline.run_merge()
if model_data is not None:
    print("Returned modeling frame:", model_data.shape)
    display(model_data.head())
    display(model_data[[
        "date", "province_name", "commodity_code", "market_level",
        "price_idr_per_kg", "mask_weather", "mask_production", "mask_satellite",
        "evidence_quality_score", "abstention_required", "shock_7d"
    ]].head(20))

## 11. Data-quality artifacts

In [ ]:
quality = pipeline.run_quality()
if quality is not None:
    display(quality["summary"])
    display(quality["coverage"].query("scope == 'national'"))
    print("Manifest checks:", quality["manifest"]["checks"])

## 13. Visualisasi diagnostik

Figures disimpan ke `reports/figures/` dan juga ditampilkan dalam notebook.

In [ ]:
figures = pipeline.run_visualization(close_figures=False)
if figures:
    import matplotlib.pyplot as plt
    plt.show()
    display(pd.DataFrame([
        {"figure": name, "path": str(path)}
        for name, (_, path) in figures.items()
    ]))

## 13. Final status dan lokasi output

In [ ]:
display(pipeline.status())

if pipeline.errors:
    print("Failed stages:", list(pipeline.errors))
    print("Detail traceback tersedia di pipeline.errors[stage] dan logs/pipeline.log")

output_rows = [{"artifact": key, "path": str(path)} for key, path in pipeline.paths.items()]
if output_rows:
    display(pd.DataFrame(output_rows).sort_values("artifact"))

ready_path = PROJECT_ROOT / "data" / "ready" / "model_dataset_eligible.parquet"
print("Primary model-ready dataset:", ready_path)
print("Exists:", ready_path.exists())

## Optional: one-cell batch run

Gunakan hanya setelah konfigurasi stabil. Stage yang sudah memiliki output dapat di-resume atau membaca file existing pada tahap berikutnya.

In [ ]:
# Fresh orchestrator for a complete batch run:
# batch_pipeline = PasarPulsePipeline(config)
# batch_status = batch_pipeline.run_all(close_figures=False)
# display(batch_status)